<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Background Subtraction</b></h1>
</div>

## Theoretical Foundations

This notebook develops the background-subtraction project from first principles while preserving exactly the same 13-task order as the executable Implementation notebook.

### Big Picture

The problem is sparse moving-tool detection in a mostly static fluoroscopic sequence. The method separates temporal change from persistent anatomy, then progressively suppresses nuisance variation before segmentation.

### Core Background-Subtraction Model

For frame (I_t) and background estimate (B),

[
D_t = I_t-B.
]

A sequence of intensity normalization, spatial filtering, spectral filtering, morphology, and segmentation converts (D_t) into a binary tool/background mask.

### Canonical Notation Used Everywhere

| Symbol | Meaning |
|---|---|
| (I_t) | fluoroscopic frame at time (t) |
| (B) | background estimate |
| (D_t) | background-subtraction residual |
| (G_t) | manual ground-truth mask |
| (M_t) | predicted binary mask |
| (sigma) | Gaussian spatial-filter scale |
| (D_0) | spectral cutoff |
| (T) | segmentation threshold |
| (Q) | field-of-view mask |

### Expected Learning Outcomes

By the end you should be able to explain why each retained processing stage exists, how controlled ablation isolates its effect, why the GMM branch can be rejected despite being more sophisticated, and how quantitative metrics plus anatomical guardrails determine the final pipeline.

## 1. Validate Data and Ground-Truth Paths

Background subtraction is only meaningful when image and annotation paths are paired correctly. A frame and its masks must share spatial dimensions. The manual guidewire and microcatheter annotations are combined into one tool reference.

### Concept Check

Why is a path check part of the scientific method rather than only a software convenience? Because a silently mismatched frame/mask pair can produce numerically valid but scientifically meaningless metrics.

## 2. Reproduce the Original Baseline Pipeline

The original method treats the first processed frame as the background. For processed frame (I_t) and reference (B), the signed residual is

[
R_t = -(I_t-B).
]

That baseline establishes a reproducible point of comparison. Improvements must be evaluated against it rather than replacing it without evidence.

### Concept Check

Why reproduce the original first? Without a baseline, later gains cannot be attributed to a specific change.

## 3. Replace the First-Frame Background with a Temporal Median

A temporal median estimates a static background robustly:

[
B_{mathrm{med}}(x,y)=operatorname{median}_t I_t(x,y).
]

Sparse moving tools affect a pixel only during some frames, so the median can suppress them while retaining persistent anatomy.

### Concept Check

When would a temporal median fail? If the foreground occupies the same pixel for most frames or if the background itself changes strongly over time.

## 4. Stabilize the Pre-Subtraction Histogram Transformation

Frame-dependent normalization can create artificial temporal differences. A fixed mapping uses limits (L,H) estimated once from the background:

[
I'(x,y)=operatorname{clip}left(rac{I(x,y)-L}{H-L},0,1ight).
]

Applying the same mapping to every frame preserves temporal comparability.

### Concept Check

Why can independent per-frame contrast stretching harm subtraction? Because identical physical intensities can be mapped differently in different frames.

## 5. Optimize Spatial Gaussian Filtering

Gaussian smoothing suppresses high-frequency acquisition noise before subtraction. For standard deviation (sigma),

[
G_sigma(x,y)=rac{1}{2pisigma^2}expleft(-rac{x^2+y^2}{2sigma^2}ight).
]

Too little smoothing leaves noise; too much smoothing can erase thin guidewires.

### Concept Check

Why must the background be rebuilt for each candidate sigma? The current frame and reference must live in the same filtered domain.

## 6. Add and Tune Spectral-Domain High-Pass Filtering

Frequency-domain filtering uses

[
F(u,v)=mathcal{F}{R(x,y)},qquad G(u,v)=H(u,v)F(u,v).
]

A Gaussian high-pass response can suppress slowly varying residual background while preserving sharper tool responses. Its cutoff controls the trade-off between background removal and tool preservation.

### Concept Check

Why is an aggressive high-pass filter dangerous? Thin tools are high-frequency structures, but so are noise and anatomical edges.

## 7. Optimize Morphological Refinement

Mathematical morphology changes binary or grayscale structures according to a structuring element. Dilation expands selected responses; opening removes small structures through erosion followed by dilation.

The structuring-element radius therefore has a direct geometric meaning relative to catheter/guidewire width.

### Concept Check

Why is morphology evaluated after upstream filtering is fixed? Otherwise the apparent benefit could be caused by another simultaneous change.

## 8. Optimize the Segmentation Threshold

Threshold segmentation maps a feature image (R) to a binary decision:

[
M_{mathrm{tool}}(x,y)=mathbf{1}[R(x,y)>T].
]

The threshold is a model parameter. It must be tuned after preprocessing because every retained preprocessing change alters the feature distribution.

### Concept Check

Why is threshold sensitivity more informative than one hand-picked threshold? It reveals whether performance is robust or depends on a fragile operating point.

## 9. Compare Threshold Segmentation with EM/GMM

A two-component Gaussian Mixture Model represents the feature density as

[
p(x)=sum_{k=1}^{2}pi_k,mathcal{N}(xmidmu_k,Sigma_k).
]

EM alternates between estimating component responsibilities (E-step) and updating component parameters (M-step). The higher-mean component is treated as tool-like.

A GMM is not guaranteed to outperform thresholding: sparse tools create strong class imbalance, and the second Gaussian may model anatomical residuals rather than the desired object.

### Concept Check

Why can a simpler threshold win? Because model complexity is useful only when its assumptions match the data distribution.

## 10. Restrict Processing to a Valid Field-of-View Mask

A field-of-view mask removes pixels that are physically outside the useful fluoroscopic support. It must never improve metrics by deleting true tool pixels.

For candidate ROI (Q),

[
mathrm{Coverage}_t=rac{|Qcap G_t|}{|G_t|},
]

where (G_t) is the annotated tool set. A safe ROI requires full ground-truth coverage on every frame.

### Concept Check

Why is coverage a guardrail? It prevents metric improvement through invalid exclusion of difficult true positives.

## 11. Assemble the Final Retained Pipeline

The final pipeline is the composition of only retained decisions. In order:

[
	ext{frame}
ightarrow 	ext{Gaussian}
ightarrow 	ext{fixed contrast map}
ightarrow 	ext{temporal background subtraction}
ightarrow 	ext{spectral HPF}
ightarrow 	ext{morphology}
ightarrow 	ext{threshold}
ightarrow 	ext{FOV}
ightarrow 	ext{binary mask}.
]

A rejected GMM experiment remains documented evidence but is not part of this chain.

### Concept Check

Why separate ablation history from the final pipeline? Reproducibility requires one unambiguous retained method.

## 12. Compute Sequence-Level Quantitative Evaluation

For reference mask (G) and predicted mask (M), the project reports pixel-domain differences.

[
mathrm{SAD}=rac{1}{N}sum_i |G_i-M_i|,
qquad
mathrm{MSE}=rac{1}{N}sum_i(G_i-M_i)^2.
]

For peak value (L),

[
mathrm{PSNR}=20log_{10}left(rac{L}{sqrt{mathrm{MSE}}}ight).
]

Sequence-level mean and standard deviation summarize central performance and variability.

### Concept Check

Why retain per-frame curves in addition to means? A good mean can hide catastrophic failure on one intervention frame.

## 13. Run Numerical and Output-file Validation Checks

Validation is a scientific contract. It checks that inputs exist, outputs have valid shapes and values, metrics are finite and complete, retained parameters are defined, the FOV does not remove annotations, and expected figures exist.

A notebook that merely reaches the last cell is not sufficient evidence of correctness.

### Concept Check

What is the difference between execution and validation? Execution means the code ran; validation checks whether the result satisfies the intended numerical and scientific constraints.

## Expected Competencies

### Level 1 — Beginner

You can explain background subtraction, binary masks, manual annotations, and the meaning of SAD/MSE/PSNR.

### Level 2 — Operational

You can reproduce the baseline, build a temporal-median background, apply fixed contrast mapping, Gaussian filtering, morphology, thresholding, and sequence-level evaluation.

### Level 3 — Advanced

You can design controlled ablations, reason about spatial/spectral trade-offs, compare thresholding with EM/GMM, and detect metric improvements that violate anatomical validity.

### Level 4 — Advanced Understanding

You can reason about class imbalance, background-model assumptions, FOV guardrails, temporal stability, metric limitations, and how this classical pipeline could be extended without invalidating experimental attribution.

### Complete Dependency Chain

[
oxed{
	ext{frames + annotations}
ightarrow
	ext{baseline}
ightarrow
B_{mathrm{median}}
ightarrow
	ext{fixed mapping}
ightarrow
	ext{spatial filter}
ightarrow
	ext{spectral filter}
ightarrow
	ext{morphology}
ightarrow
	ext{segmentation}
ightarrow
	ext{FOV}
ightarrow
	ext{metrics + overlays}
}
]

If you can justify every arrow and state which experiments were rejected, you understand the complete project pipeline.

## Scope and Limitations

### Included

- static/mostly-static background modeling;
- temporal median reference;
- histogram transformation;
- spatial Gaussian filtering;
- spectral high-pass filtering;
- morphology;
- deterministic thresholding;
- EM/GMM comparison;
- FOV masking;
- SAD, MSE, and PSNR evaluation;
- qualitative mask/overlay inspection.

### Not included

- non-rigid registration;
- learned segmentation networks;
- adaptive online background models;
- optical flow;
- uncertainty calibration;
- clinical validation or deployment.